In [64]:
# sets up the necessary imports, selects the appropriate computation device, and defines key configuration parameters for a machine learning task using PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)
block_size = 8
batch_size = 4
max_iters = 1000
learning_rate = 3e-4
dropout = 0.2
eval_iters = 250

mps


In [65]:
# Reading the contents of a text file and processing it to determine the number of unique characters in the text
with open('arte_de_amar.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(set(text)))
print(chars)
vocab_size = len(chars)

['\n', ' ', '!', '#', '(', ')', ',', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'Y', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'x', 'y', 'z', '¡', '¿', 'Á', 'É', 'Í', 'Ú', 'á', 'é', 'í', 'ñ', 'ó', 'ú', 'ü', '\ufeff']


In [66]:
# Encode a text into numerical format using a character-level encoding scheme 
# and then converting it into a PyTorch tensor.
string_to_int = { ch: i for i, ch in enumerate(chars) }
int_to_string = { i: ch for i, ch in enumerate(chars) }

# Define two lambda functions to encode and decode the text
encode = lambda s: [string_to_int[ch] for ch in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

# Convert the text into a PyTorch tensor
data = torch.tensor(encode(text), dtype=torch.long)

print(data.size())
print(data[:100])

torch.Size([138559])
tensor([85, 25, 58,  1, 47, 64, 66, 51,  1, 50, 51,  1, 47, 59, 47, 64,  0,  0,
        21, 67, 66, 54, 61, 64, 18,  1, 35, 68, 55, 50,  0,  0, 38, 51, 58, 51,
        47, 65, 51,  1, 50, 47, 66, 51, 18,  1, 33, 47, 70,  1,  9,  6,  1, 10,
         8, 10, 10,  1, 44, 51, 22, 61, 61, 57,  1,  3, 14, 15, 17, 14,  9, 45,
         0,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 33,
        61, 65, 66,  1, 64, 51, 49, 51, 60, 66])


In [67]:
# Preparing data for training and validation in a machine learning task using PyTorch
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

# Generates a batch of data for either training or validation, depending on the value of the split parameter.
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('input:', x)
print('target:', y)

input: tensor([[47, 65,  1, 62, 47, 64, 66, 51],
        [51, 58, 58, 47, 50, 61, 65,  0],
        [ 1, 66, 55, 51, 64, 64, 47,  1],
        [51,  1, 48, 47, 56, 51, 58,  6]], device='mps:0')
target: tensor([[65,  1, 62, 47, 64, 66, 51, 65],
        [58, 58, 47, 50, 61, 65,  0, 47],
        [66, 55, 51, 64, 64, 47,  1, 78],
        [ 1, 48, 47, 56, 51, 58,  6,  1]], device='mps:0')


In [68]:
# Examining sequences from the training data to understand the relationship between input contexts and their corresponding targets.
# x = train_data[:block_size]
# y = train_data[1:block_size+1]

# for t in range(block_size):
#     context = x[:t+1]
#     target = y[t]
#     print('when input is', context, 'target is', target)

In [69]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [70]:
# Perform the forward pass to compute logits and loss, generate new tokens based on the model
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        
    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)
        
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, index, max_new_tokens):
        # index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(index)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1)
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)
        


Í[hq.rz,7éóúAzl9aókláeD érPMñMosR;OüoCUógÉL3ñBKbUgh;ó?﻿b8é:x!c_#V  SnESníSóQükDV
CLt7;A5 É4VógÁFíeú#éM﻿3E(áo_B﻿3z e]oiÍfQéf#U!é:KfrmiuÍU6[ÚR4Í#H2(6t9Yt9déHlmz(AtÚUíO_Í9:rvbi)ak?VL;;TR)JEé3éBtKzaÉ[üGd¡UV0Mé:r?iL[)Kf¿hkPó0F ?u
oJ9jJH]V8 pNSóeD4ahL(jbFgpNÍ6[)J!
KP
éEqrYqM._0k3﻿g!óáíhÁL:bhhLfqjkiL,:p﻿cB:¿:,!tEQx.rU0Á]í
Íá6#U6cIóUme;RÉmóiOQ﻿S(02ó,LÍcflÍ0SYÚqkrL(fyQKIn:4IÁ.AÉ:?﻿k?Yed¿:Dózólv¡l20kCtex03ie;viOapGMLY7hK8q
vQM,D[Úvbr
u﻿B(3czaQICb7íGSsoFP¡Eá6üOs9eI2sy5NLMl3cÉc]3U;#KvqéBE(6r 3ddTfERx,k#s94¡

oahHkáO0IH[p82U)¿dFFÉüqj!__lAijzí c:C,.NSt¡;ÚüU
] m6#U48léaíVIi]BvúpjHrNSnjt:Cvú¡:KM;:ó?a.kNf]Cu1I54C,﻿aO)áCÚÉdáYÁ?m#Vñf2ÁkNúACñC(q2AO]¿6[ÚYDévQ
4l5N:pQ!(mz1uJOVíé#úaOpj0PÁk[ñERNvÍÍ!﻿ÉAVd,¿JIérÁkPoG NSk9sqé)L]9bü5hMyíEJGhL)bíTqé.C01;Úrñ﻿rñTdñ9,SYEu¡kñO¡:yQ﻿híh7í h;íM[AjÁmüñp#548ómih,5pó9;Í4vvPÉOÚrmf;íh:m]3.y¡lPmfkPyQJ!yüYYQVHuJ3E(461,OD:¡Vd hís#úAmHhKczaQ﻿,JT]igd)1RíOBÁI4áuóy¡rúyYÁ,5ü.uSk8NJdPMxlüKj#Ú;,gyL;áBóRFH#1n(NbfrU4Dy[.OTA0ÁjDYqlÉÉ ¡zúVpHlP
e_é3fa8Pósk!jo!HBb7ón2Q[(oopNÍJ;q0)¿ññTóo:DÉ_

In [71]:
#  Create optimizer1 and train the model
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss: 4.912, val loss: 4.899
step: 250, train loss: 4.835, val loss: 4.849
step: 500, train loss: 4.777, val loss: 4.776
step: 750, train loss: 4.718, val loss: 4.739
4.729068756103516


In [72]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


﻿h;
50FL]_u]__BóDÉx9.óJV
?4z ?k:2b7o[Ú﻿?antL3:Hlm,P1py¡é#xIÁbJhG?ÍjQ.A3﻿82mL5Hh;NzKvfcr.r¿ñCIñ4ákr;xoCll4sF:A¿7)o70Í#U)EáyCV!ú68JJaHl7lo9L(Tox]y5pNOP#¿Mérú9NuE2Jóyp(4ágy]C6ÚlÁh;é:0Á,.telHrÚE9b8MA5HM¿xÁn#VHddvKIB 2ó!,AáBrh;;:﻿3B]E7PCEKzOMÁá6üá0pa﻿phLÉESkhp﻿áoÉ4fH#oLHhGt﻿rv[]3ÁyySB QT¿rctG?0;5NY[
JGü5V#VH2ólmJ﻿Dm
qaRHÍ7¿z)áú99VÍ._0ÚR]ñx¿7G;Ah(T_YST:#L3yLl i) yíq#y¡üq!dH:6é!tC,3Áa﻿fR54cÉADPB 34)yEQ﻿CSxúR?m6 ÉEDÉGAKA]lO
uSnG?;Í(UcáL(5Grñ8c.hL!e_,¿JYYm6U1jJ#óeOú¡vfFz4JzBÚHpex¿O]VYnnl5úíhK,Mj2S]t6]AEh
